# Diderot ML Lab — Change Impact & Regression Evidence

This notebook turns the Test Authority method into an executable experiment.

The central boundary is `G_observed != G_true`: selectors R1–R5 may use only the imperfect engineering view, historical evidence and test metadata. `G_true` is reserved for simulation and post-hoc scoring.

**The system is synthetic and does not represent an IN Groupe production architecture.**

We compare R0 full-suite, R1 history, R2 code/control graph, R3 multi-layer system graph, R4 risk-aware system graph and R5 learned historical-impact augmentation.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO = 'https://github.com/gharbonnier78/diderot-machine-learning-specialization.git'
REFS = ['main', 'lab/change-impact-regression-simulator']
SENTINEL = Path('labs/change-impact-regression/scenarios/identity_platform.yaml')

if not Path('pyproject.toml').exists() or not SENTINEL.exists():
    repo_dir = Path('diderot-machine-learning-specialization')
    if repo_dir.exists(): shutil.rmtree(repo_dir)
    selected_ref = None
    for ref in REFS:
        subprocess.run(['git','clone','--depth','1','--branch',ref,REPO,str(repo_dir)], check=True, stdout=subprocess.DEVNULL)
        if (repo_dir / SENTINEL).exists():
            selected_ref = ref
            break
        shutil.rmtree(repo_dir)
    if selected_ref is None: raise RuntimeError('Lab not found on main or review branch')
    os.chdir(repo_dir)
    print('Using Git ref:', selected_ref)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
print('Ready:', Path.cwd())

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from diderot_mls.change_impact import (
    InjectedFault, build_fault_catalogue, evaluate_fault_campaign,
    load_scenario, summarize_fault_campaign,
)
from diderot_mls.change_impact.experiment import run_comparison, monte_carlo_sweep
from diderot_mls.change_impact.visualization import plot_system_graph, plot_selection_overlay

scenario = load_scenario(Path('labs/change-impact-regression/scenarios/identity_platform.yaml'))
print(scenario.name, len(scenario.nodes), 'nodes,', len(scenario.edges), 'edges,', len(scenario.tests), 'tests')
print('Changes:', ', '.join(scenario.changes))

## 1. One controlled change

Start with a cache-TTL modification. Functional retest may pass while shared state propagates toward authorization behavior. The engineering graph is intentionally incomplete.

In [ ]:
bundle = run_comparison(scenario, change_id='CHG_CACHE_TTL', completeness=0.78, false_edge_rate=0.03, budget=22.0)
cols = ['execution_cost','tests_executed','impact_recall','critical_impact_recall','impacted_node_coverage','critical_mean_pod','critical_min_pod','critical_regression_miss_rate']
bundle.results[cols].round(3)

`impacted_node_coverage` and `critical_mean_pod` are deliberately separate. A selected test can exercise an impacted element yet have a weak oracle for the failure mode. This is an executable form of **Coverage Isn't Enough**.

In [ ]:
fig, ax = plt.subplots(figsize=(14,9))
plot_system_graph(scenario, bundle.observed_graph, title='Engineering view available to selectors (G_observed)', ax=ax)
plt.show()

## 2. Reveal truth only after selection

The next plots are post-hoc evaluation. Larger nodes are covered by at least one selected test; coverage still does not guarantee detection.

In [ ]:
for label in ['R2','R4','R5']:
    fig, ax = plt.subplots(figsize=(14,9))
    plot_selection_overlay(scenario, bundle.observed_graph, bundle.current_outcome, bundle.selections[label], title=f'{label} — {bundle.selections[label].strategy}', ax=ax)
    plt.show()

## 3. Degrade architecture knowledge

Vary `G_observed` completeness and measure the resulting critical-impact recall. This tests a systems-engineering question: how much assurance is lost because propagation structure is poorly known?

In [ ]:
sweep = monte_carlo_sweep(scenario, change_id='CHG_EVENTBUS_POOL', completeness_values=(0.45,0.60,0.75,0.90,1.00), repetitions=8, budget=20.0)
summary = sweep.groupby(['completeness','label'])[['critical_impact_recall','critical_mean_pod','execution_cost']].mean().reset_index()
fig, ax = plt.subplots(figsize=(10,6))
for label, group in summary.groupby('label'):
    ax.plot(group['completeness'], group['critical_impact_recall'], marker='o', label=label)
ax.set(xlabel='Observed architecture completeness', ylabel='Mean critical impact recall', title='Knowledge completeness vs critical impact recall', ylim=(-0.02,1.02))
ax.legend(); plt.show()

## 4. Coverage vs detection power

Run the supplier-change case and compare impacted-node coverage with critical POD. R5 is intentionally a simple logistic-regression baseline trained on past labelled changes; the current hidden outcome is excluded. A negative result for R5 is valid.

In [ ]:
one = run_comparison(scenario, change_id='CHG_SUPPLIER_SDK', completeness=0.72, false_edge_rate=0.03, budget=20.0, graph_seed=31, propagation_seed=41, detection_seed=51)
fig, ax = plt.subplots(figsize=(8,6))
for label, row in one.results.iterrows():
    ax.scatter(row['impacted_node_coverage'], row['critical_mean_pod'], s=80)
    ax.annotate(label, (row['impacted_node_coverage'], row['critical_mean_pod']), xytext=(5,5), textcoords='offset points')
ax.set(xlabel='Impacted-node coverage', ylabel='Critical mean POD', title='Coverage and detection are different evidence planes', xlim=(-0.02,1.02), ylim=(-0.02,1.02))
plt.show()
one.results[cols].round(3)

## 5. Explicit fault injection: covered does not mean detectable

Now separate the mutation/fault-injection plane from change propagation. The first probe deliberately injects a failure mode on a node that one selected test *does cover*, but for which that test has no relevant oracle. Expected result: `covered=True` while `pod=0`. This is not a rhetorical example; it is computed from the test/failure-mode contract.

In [ ]:
from diderot_mls.change_impact.selectors import SelectionResult

oracle_gap_selection = SelectionResult(
    strategy='oracle_gap_probe',
    selected_tests=['T_UNIT_IDENTITY'],
    predicted_impacts={'identity_service'},
)
oracle_gap_fault = InjectedFault(
    id='F_identity_authorization',
    node='identity_service',
    failure_mode='authorization',
    criticality=5.0,
)
evaluate_fault_campaign(scenario, oracle_gap_selection, [oracle_gap_fault], seed=7)

In [ ]:
critical_faults = build_fault_catalogue(scenario, critical_only=True)
fault_rows = []
for label in ['R0','R2','R4','R5']:
    frame = evaluate_fault_campaign(scenario, bundle.selections[label], critical_faults, seed=100 + int(label[1]))
    row = {'label': label, **summarize_fault_campaign(frame)}
    fault_rows.append(row)
import pandas as pd
pd.DataFrame(fault_rows).set_index('label').round(3)

## Next experiments

Try all four changes, vary budget/completeness/false-edge rate/history size, and preserve negative results. Explicit fault injection is now present in v0; the next methodological increments are environment representativity, observability, richer correlated/multi-fault and temporal/resource simulation, cross-topology validation of R5, then only later a Gymnasium interface for sequential evidence acquisition.